<a href="https://colab.research.google.com/github/shashank3110/genAI/blob/main/business_assistant_langraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.6
    Uninstalling langgraph-1.1.6:
      Successfully uninstalled langgraph-1.1.6


In [ ]:
pip install -U langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 6.2 MB/s eta 0:00:00


In [ ]:
import os
os.environ["GOOGLE_API_KEY"]="" # ADD API key

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph
from typing import TypedDict, List
from langchain_core.messages import BaseMessage
import pandas as pd

## Load & Explore knowledge base

In [ ]:
df_b = pd.read_csv('/content/clothing_business_dataset.csv')
df_s = pd.read_csv('/content/clothing_schema_dataset.csv')


In [ ]:
df_b.head()

,Category,Name,Description
0,Fact Table,Fact_Sales,Tracks product-level sales transactions
1,Fact Table,Fact_Inventory,Tracks stock levels per product/location
2,Fact Table,Fact_Orders,Tracks order lifecycle
3,Fact Table,Fact_Returns,Tracks returned items
4,Fact Table,Fact_Marketing_Performance,Tracks campaign performance


In [ ]:
df_b['Category'].unique()

array(['Fact Table', 'Dimension Table', 'KPI', 'Stakeholder'],
      dtype=object)

In [ ]:
df_b[df_b['Category']=='KPI']

,Category,Name,Description
11,KPI,Revenue,Total sales before returns
12,KPI,Gross Profit,Revenue minus cost
13,KPI,Conversion Rate,Orders divided by visitors
14,KPI,Inventory Turnover,COGS divided by avg inventory
15,KPI,Return Rate,Returned items divided by sold items
16,KPI,Customer Lifetime Value,Total revenue per customer


In [ ]:
df_b[df_b['Category']=='Stakeholder']

,Category,Name,Description
17,Stakeholder,Sales Department,Sales roles and KPIs
18,Stakeholder,Supply Chain,Inventory and logistics roles
19,Stakeholder,Marketing,Campaign and growth roles
20,Stakeholder,Finance,Financial oversight roles
21,Stakeholder,Product/Merchandising,Product and design roles


In [ ]:
df_s.head(10)

,table_name,column_name,data_type,key_type
0,Fact_Sales,sales_id,BIGINT,PK
1,Fact_Sales,order_id,BIGINT,FK
2,Fact_Sales,product_id,INT,FK
3,Fact_Sales,total_amount,"DECIMAL(10,2)",NaN
4,Dim_Product,product_id,INT,PK
5,Dim_Product,product_name,VARCHAR(255),NaN
6,Dim_Product,category,VARCHAR(100),NaN
7,Dim_Customer,customer_id,INT,PK
8,Dim_Customer,first_name,VARCHAR(100),NaN
9,Dim_Customer,country,VARCHAR(100),NaN


In [ ]:
# df_s[df_s['table_name']=='Stakeholder']
df_s['table_name'].unique()

array(['Fact_Sales', 'Dim_Product', 'Dim_Customer'], dtype=object)

## Initialize LLM

In [ ]:

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # or gemini-1.5-flash
    temperature=0
)

In [ ]:
llm.invoke(f'You are business assitant and uou answer business questions using given context. [Question]: Who are the key stakeholders for the business? . [Context] {df_b} ')

AIMessage(content='Based on the provided context, the key stakeholders for the business are:\n\n*   Sales Department\n*   Supply Chain\n*   Marketing\n*   Finance\n*   Product/Merchandising', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dca36-229d-79d3-b2f3-08482d759a71-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 445, 'output_tokens': 77, 'total_tokens': 522, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 37}})

In [35]:
# Define tools

@tool
def rag_answer(question: str) -> str:
    """
    Answer User's business questions using provided Context.

    Args:
        question: User question: str

    """

    prompt = f"""You are business FAQ assitant and you answer business questions
                  using given context. [Question]: {question} . [Context] {df_b} """

    response = llm.invoke(prompt)
    metadata = {'links': ['abc12x.com']}

    # Return a dictionary with content and metadata directly
    return {'response': response.content, 'metadata': metadata }


@tool
def sql_answer(question: str) -> str:
    """
    Generate SQL code to answer User's business questions using provided Context.

    Args:
        question: User question: str

    """

    prompt = f"""You are SQL assitant and you answer business questions with valid SQL code
                  using given context which contains the schema for the various
                  tables. [Question]: {question} . [Context] {df_s} """


    return llm.invoke(prompt)

In [36]:
# initialize tools
tools = [rag_answer, sql_answer]
tool_names = [tool.name for tool in tools]
llm_with_tools = llm.bind_tools(tools)

In [37]:
class AgentState(MessagesState):
    pass



In [38]:

def agent(state: AgentState):
    if not state["messages"]:
        raise ValueError("Empty state!")

    response = llm_with_tools.invoke(state["messages"])
    print(f"[DEBUG] MODEL OUTPUT: {response}")
    print("[DEBUG] TOOL CALLS:", getattr(response, "tool_calls", None))

    return {"messages":  state["messages"] + [response]}


In [39]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)


In [40]:
# for conditional branching/routing


def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return "end"


In [41]:
#DEBUG CODE
def tool_node_debug(state):
    print("\n🔥 TOOL NODE INPUT:", state)

    result = tool_node.invoke(state)

    print("\n🔥 TOOL NODE OUTPUT:", result)

    return result

In [ ]:
# DEBIUG CODE
def debug_node(name, fn):
    def wrapper(state):
        print(f"\n🔥 {name} INPUT:")
        for m in state["messages"]:
            print(type(m), m)

        result = fn(state)

        print(f"\n🔥 {name} OUTPUT:")
        for m in result.get("messages", []):
            print(type(m), m)

        return result
    return wrapper


## Build Agent execution control graph

In [42]:
business_graph = StateGraph(AgentState)
business_graph.add_node("agent", agent)
business_graph.add_node("tools",tool_node)
# math_graph.add_node("tools",tool_node_debug)

# math_graph.add_node("agent", debug_node("AGENT", agent))
# math_graph.add_node("tools", debug_node("TOOLS", tool_node_debug))


business_graph.set_entry_point("agent")

business_graph.add_conditional_edges("agent", should_continue,  {
        "tools": "tools",
        "end": "__end__"
    }
    )

business_graph.add_edge("tools", "agent")
# math_graph.add_edge("tools", "end")

In [43]:
# compile agentic graph
business_agent = business_graph.compile()

In [44]:
from langchain_core.messages import HumanMessage, SystemMessage


In [49]:
result = business_agent.invoke(

     {
          "messages": [
              SystemMessage(content="You are a Business assistant who answers business related questions through text or SQL code. Always use tools for answering business FAQ questions or providing SQL code."),
              HumanMessage(content="who are the key stakeholders?"),
              # HumanMessage(content="How can i calculate gross profit?"),
              # HumanMessage(content="Define inventory turnover and customer life time value?"),
              # HumanMessage(content="Calculate total sales per product "),
              # HumanMessage(content=" Get total customers per country or region?")

              ]
     }

)

[DEBUG] MODEL OUTPUT: content='' additional_kwargs={'function_call': {'name': 'rag_answer', 'arguments': '{"question": "who are the key stakeholders?"}'}, '__gemini_function_call_thought_signatures__': {'4ab331b9-7223-4e80-9999-cbc71491e2fa': 'CooDAQw51scuNzXugLYHIgRQlVKZ3vLaF0za4MZVV0hAUJzzUMdiWlK9egAOqOck16VEbEgP/31ov5UBhf366FvRtRiE+ugOuzsiDynVaV8uzUHrXr+iFFSbsfyE298kKoSTE98DeIgztp0BPwNmBH+OS1XVhLceUkWWazq2OvDEFv7wK/sgkpPy4jreNF3hwSH2zDybQgJXreSz/7dJHLMmnc5VpYIF81I8giews2p4HxpyWX7hbiXxJcgTAKUQOd7vpLduzs2jy+14GVS0VDrsu6ALQeaBhRbNHd2Fe3i3Wce3pJ7EvxWXwSX2XTcYSmEE2QIVooaf0A+h6JVex3p8SdLz7+blfjEF4PDbkGj558zt8y3dA/wJ+I5LP+RpEbCk6Ktc/LEnpawA21D0yEba5ahEnx9HaxgelvdSQQi/ia9+WjLRRUPnCzVwtN20Vsse81KHnH8FqjCNls40YmgAwBnaYORVinahC6EXupVYdErxsPL6M+Ls+vQFKhqcXmlrp3Mu2zlG9klK4Q=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019dca4d-6e96-7360-b2ee-ecf9a83028b4-0' tool_calls=[{'name': 'rag_answer',

In [46]:
print(result["messages"][-1].content)

The key stakeholders are:
*   Sales Department
*   Supply Chain
*   Marketing
*   Finance
*   Product/Merchandising


In [51]:
print(result["messages"][3])

content='{"response": "The key stakeholders are:\\n*   Sales Department\\n*   Supply Chain\\n*   Marketing\\n*   Finance\\n*   Product/Merchandising", "metadata": {"links": ["abc12x.com"]}}' name='rag_answer' id='65c7a885-d33b-4cc9-a384-a10c6f310fcb' tool_call_id='4ab331b9-7223-4e80-9999-cbc71491e2fa'


In [52]:
len(result["messages"])

5

In [58]:
result["messages"][3]

ToolMessage(content='{"response": "The key stakeholders are:\\n*   Sales Department\\n*   Supply Chain\\n*   Marketing\\n*   Finance\\n*   Product/Merchandising", "metadata": {"links": ["abc12x.com"]}}', name='rag_answer', id='65c7a885-d33b-4cc9-a384-a10c6f310fcb', tool_call_id='4ab331b9-7223-4e80-9999-cbc71491e2fa')

In [33]:
result["messages"][-1].tool_calls

[]

In [48]:
result_new = business_agent.invoke(
    {
        "messages": [
            SystemMessage(content="You are a Business assistant who answers business related questions through text or SQL code. Always use tools for answering business FAQ questions or providing SQL code."),
            HumanMessage(content="who are the key stakeholders?")
        ]
    }
)
print("New result from agent invocation:")
print(result_new["messages"][3]) # This should be the ToolMessage

[DEBUG] MODEL OUTPUT: content='' additional_kwargs={'function_call': {'name': 'rag_answer', 'arguments': '{"question": "who are the key stakeholders?"}'}, '__gemini_function_call_thought_signatures__': {'12e517d4-7534-41d4-aba2-9a9448016f8e': 'CooDAQw51sf80Rwr47eXeargUs2q4E54Rky+E37/25RJty7xEurUVfWPslgsXBnIbKCarEsNaYxJTD+ONa8HDXcRQmOLGjGCDTzsXQJxeIZVF5rW85/gJTxUTfThskI2j78dAoU7eWOW49o4b+UT+BW7bTNvVUUb94vZrM+lRBXddCU7hWpI2+w3tM22ahn4wGKAigCMEZPXn286V9C+n90dnqhojplXsXCjihCDRW8Vlmoa5kvmlxCYtr0BhS6qjGm3DEetvCRI0zS3uGHeoJ3Fkzu2L+egGf9Sm/3+8jQF7DN97+vfn5WwY4V5k5iM4ze3a5pWPcCIii91Vs7J2doxcJy7hsqdgvQ6TfR8GLSGFpRfSYXRlxtCFsKEMNHBQLZGu/O/GvUSbtd9QHX4qT3JavC2JsM8ECkbD3w+GNjW5BkEazPMlaxR3l5TGkqq1qWlDJ+Uda400pNOPRdZe70OcO8p9DlgJY+wbUdiRJaiUZLmw0S+YQho9DoyAjJoSO8QV9D8rIDVNNBlKg=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019dca4c-a691-7f13-bd1e-d19539222505-0' tool_calls=[{'name': 'rag_answer',

### Extracting Metadata from Current Output

Since the `rag_answer` tool returns a dictionary containing an `AIMessage` object and your custom `metadata`, the entire output is stringified and placed in the `content` field of a `ToolMessage`. Parsing this string directly requires a regular expression to isolate the metadata dictionary, as `ast.literal_eval` cannot directly parse the `AIMessage` object within the string.

In [34]:
import re
import ast

# The ToolMessage, which contains the output of the rag_answer tool, is at index 3 in the messages list.
tool_message = result["messages"][3]
content_str = tool_message.content

# Use a regular expression to find and extract the metadata dictionary string
# It looks for `'metadata':` followed by a dictionary-like structure.
metadata_match = re.search(r"'metadata': ({[^}]+})", content_str)

if metadata_match:
    metadata_str = metadata_match.group(1)
    # Safely evaluate the extracted string to a Python dictionary
    extracted_metadata = ast.literal_eval(metadata_str)
    print("Extracted Metadata from current output:", extracted_metadata)
else:
    print("Metadata not found in the ToolMessage content.")

Extracted Metadata from current output: {'links': ['abc12x.com']}


In [61]:
type(extracted_metadata), extracted_metadata.keys(), extracted_metadata

(dict, dict_keys(['links']), {'links': ['abc12x.com']})

### Improving Metadata Handling

To make metadata extraction more straightforward, you can modify the `rag_answer` tool to return a simpler dictionary where the `response` is just the content string, and `metadata` is a separate dictionary. This way, the entire `ToolMessage.content` will be a string that `ast.literal_eval` can directly parse without complex regex.

In [31]:
dir(result["messages"][-1])

['__abstractmethods__',
 '__add__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_

In [28]:
result["messages"][-1]

AIMessage(content='The key stakeholders are:\n*   Sales Department\n*   Supply Chain\n*   Marketing\n*   Finance\n*   Product/Merchandising', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dca3f-795b-76b0-99ec-6bd597259c8d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 433, 'output_tokens': 31, 'total_tokens': 464, 'input_token_details': {'cache_read': 0}})

In [24]:

for answers in result["messages"][-1].content:

  for answer in answers.split('\n\n'):
    print('-------')
    print(answer)
    print('-------')

-------
The key stakeholders are:
*   Sales Department
*   Supply Chain
*   Marketing
*   Finance
*   Product/Merchandising
-------
-------
Gross Profit is calculated as Revenue minus cost.
-------
-------
Based on the provided context:
-------
-------
*   **Inventory Turnover** is defined as COGS (Cost of Goods Sold) divided by average inventory.
*   **Customer Lifetime Value** is defined as the total revenue per customer.
-------
-------
```sql
SELECT
  dp.product_name,
  SUM(fs.total_amount) AS total_sales
FROM Fact_Sales AS fs
JOIN Dim_Product AS dp
  ON fs.product_id = dp.product_id
GROUP BY
  dp.product_name;
```
-------


In [ ]:
# result